# импорты

In [ ]:
!pip install -q pgmpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteBayesianEstimator
import networkx as nx
from pgmpy.inference import VariableElimination

# данные

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv")

df.head()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
data = df.drop(columns=['bill_depth_mm', 'year']).copy()
data = data.dropna()
data = data.reset_index(drop=True)

data.isnull().sum()

In [ ]:
numeric_features = [
    'bill_length_mm',
    'flipper_length_mm',
    'body_mass_g'
]

for column in numeric_features:
    plt.figure(figsize=(7, 4))
    sns.histplot(data[column], bins=50)
    plt.title(f'распределение признака {column}')
    plt.xlabel(column)
    plt.ylabel('количество пингвинов')
    plt.show()

In [ ]:
data_discrete = data.copy()

cols = ['bill_length_mm', 'flipper_length_mm', 'body_mass_g']

quantile_labels = ['low', 'mid', 'large']

for col in cols:
    data_discrete[col] = pd.qcut(
        data_discrete[col],
        q=3,
        labels=quantile_labels,
        duplicates='drop'
    )

data_discrete.head()

In [ ]:
for column in numeric_features:
    print(data_discrete[column].value_counts(sort=False))
    print()

# Граф

In [ ]:
model = DiscreteBayesianNetwork([
    ('island', 'species'),
    ('species', 'bill_length_mm'),
    ('species', 'flipper_length_mm'),
    ('species', 'sex'),
    ('sex', 'bill_length_mm'),
    ('species', 'body_mass_g'),
])

In [ ]:
nx.is_directed_acyclic_graph(model)

In [ ]:
graph = nx.DiGraph()

graph.add_nodes_from(model.nodes())
graph.add_edges_from(model.edges())

pos = nx.spring_layout(graph, seed=40)

plt.figure(figsize=(12, 8))

pos = nx.spring_layout(graph, seed=20)

nx.draw_networkx_nodes(
    graph,
    pos,
    node_size=3500,
    node_color='lightblue'
)

nx.draw_networkx_labels(
    graph,
    pos,
    font_size=9,
    font_weight='bold'
)

nx.draw_networkx_edges(
    graph,
    pos,
    arrows=True,
    arrowstyle='->',
    arrowsize=30,
    width=1.8,
    edge_color='black',
    node_size=3500,
    min_source_margin=15,
    min_target_margin=25
)

plt.axis('off')
plt.show()


In [ ]:
model.fit(
    data_discrete,
    estimator=DiscreteBayesianEstimator(
        prior_type='BDeu',
        equivalent_sample_size=5
    )
)

model.check_model()

In [ ]:
for cpd in model.get_cpds():
    print(f"CPD of {cpd.variable}:")
    print(cpd)
    print("-" * 40)

# вероятностный вывод

Как меняется распределение массы тела пингвина (body_mass_g), если нам известно, что это самец (sex = male), по сравнению с ситуацией, когда пол неизвестен?

In [ ]:
infer = VariableElimination(model)

q1_base = infer.query(variables=['body_mass_g'])
print("P(body_mass_g):")
print(q1_base)

q1_evid = infer.query(variables=['body_mass_g'], evidence={'sex': 'male'})
print("\nP(body_mass_g | sex=male):")
print(q1_evid)

Какова вероятность того, что у пингвина длинный клюв (bill_length_mm = High), если известно, что он живет на острове Biscoe (island = Biscoe) и относится к виду Gentoo (species = Gentoo)?

In [ ]:
q2 = infer.query(
    variables=['bill_length_mm'],
    evidence={'island': 'Biscoe', 'species': 'Gentoo'}
)
print("P(bill_length_mm | island=Biscoe, species=Gentoo):")
print(q2)

Как изменится вероятность того, что пингвин — это Chinstrap (species = Chinstrap), если мы узнаем, что у него длинные плавники (flipper_length_mm = High) и он самец (sex = male)?

In [ ]:
q3_base = infer.query(variables=['species'])
print("P(species):")
print(q3_base)

q3_evid = infer.query(
    variables=['species'],
    evidence={'flipper_length_mm': 'large', 'sex': 'male'}
)
print("\nP(species | flipper_length_mm=large, sex=male):")
print(q3_evid)

эксперимент с evidence

In [ ]:
target = 'body_mass_g'

print("1. Без evidence P(body_mass_g):")
print(infer.query(variables=[target]))

print("\n2. Evidence: sex=male")
print(infer.query(variables=[target], evidence={'sex': 'male'}))

print("\n3. Evidence: sex=male, species=Gentoo")
print(infer.query(variables=[target], evidence={'sex': 'male', 'species': 'Gentoo'}))

print("\n4. Evidence: sex=male, species=Gentoo, bill_length_mm=low")
print(infer.query(variables=[target], evidence={'sex': 'male', 'species': 'Gentoo', 'bill_length_mm': 'low'}))

In [ ]:
import pandas as pd

# Собираем данные из наших запросов
results = []
queries = [
    ("Ничего не известно", {}),
    ("sex=male", {'sex': 'male'}),
    ("sex=male, species=Gentoo", {'sex': 'male', 'species': 'Gentoo'}),
    ("sex=male, species=Gentoo, bill=Low", {'sex': 'male', 'species': 'Gentoo', 'bill_length_mm': 'low'})
]

for name, ev in queries:
    res = infer.query(variables=['body_mass_g'], evidence=ev)
    results.append({
        'Условие': name,
        'P(Low)': res.values[0],
        'P(Medium)': res.values[1],
        'P(High)': res.values[2]
    })

df_results = pd.DataFrame(results)
display(df_results)

# analиз структуры

In [ ]:
edges_without = [
    ('species', 'island'),
    ('species', 'bill_length_mm'),
    ('species', 'flipper_length_mm'),
    ('sex', 'body_mass_g')
]

model_no_edge = DiscreteBayesianNetwork(edges_without)

model_no_edge.fit(
    data_discrete,
    estimator=DiscreteBayesianEstimator(
        prior_type='BDeu',
        equivalent_sample_size=5
    )
)

infer_no_edge = VariableElimination(model_no_edge)

print("Оригинальная сеть (с ребром species -> body_mass_g):")
print(infer.query(variables=['body_mass_g'], evidence={'sex': 'male', 'species': 'Gentoo'}))

print("\nНовая сеть (без ребра species -> body_mass_g):")
print(infer_no_edge.query(variables=['body_mass_g'], evidence={'sex': 'male', 'species': 'Gentoo'}))